# Prithvi WxC 2.3B — Fine-tuning on (partial) MERRA-2  ·  Colab Pro, max effort
### ExtremeCast · subseasonal heat-extreme forecasting

Runs a **real fine-tune** of Prithvi WxC on **real MERRA-2 data** you download from NASA GES DISC,
designed to survive Colab's session cap by persisting data + checkpoints to **Google Drive** and
**resuming across sessions**.

**Honest scope.** This fine-tunes a *frozen backbone + trainable correction head* (fits an A100 40 GB
cleanly) to improve the model's **T2M** forecast, and reports skill vs persistence and Prithvi zero-shot.
It is a defensible downstream result — not full 2.3B pretraining, and bounded by how much MERRA-2 you can
download/store. Everything is stated so you can present it honestly.

**You need:**
1. A **NASA Earthdata** account (free): https://urs.earthdata.nasa.gov/ — and approve GES DISC access:
   https://urs.earthdata.nasa.gov/approve_app?client_id=e2WVk8Pw6weeLUKZYOxvTQ
2. **Google Drive** space for the compact inputs + checkpoints (a few GB is enough to start).
3. Runtime → **A100 GPU + High-RAM**.

## 1 · GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Set Runtime -> GPU (A100 recommended)."
print("GPU:", torch.cuda.get_device_name(), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2 · Mount the 5 TB Drive via rclone (different account than Colab)
Colab's own `drive.mount` can only reach the account you're logged into Colab with. To use the 5 TB
Drive on your *other* account, we mount it with **rclone**. Data + checkpoints written here are owned by
(and count against) that 5 TB account, so nothing is lost when the session ends.

**One-time: get your rclone token** (on your laptop, which has a browser):
```
rclone authorize "drive"
```
Sign into the **5 TB account**, then copy the whole JSON it prints (looks like `{"access_token":...}`)
and paste it into `RCLONE_TOKEN` below.

In [ ]:
import subprocess, os, time
from pathlib import Path
subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True, check=False)

# Paste the JSON from `rclone authorize "drive"` (run on your laptop, signed into the 5TB account).
# You may paste it WITH or WITHOUT surrounding quotes - both work.
import json
RCLONE_TOKEN = 'PASTE_YOUR_TOKEN_JSON_HERE'
tok = json.dumps(RCLONE_TOKEN) if isinstance(RCLONE_TOKEN, dict) else str(RCLONE_TOKEN).strip()
assert 'PASTE_YOUR_TOKEN' not in tok, "Paste your rclone token first (see markdown above)."

conf = '[gdrive5tb]\ntype = drive\nscope = drive\ntoken = ' + tok + '\n'
os.makedirs('/root/.config/rclone', exist_ok=True)
open('/root/.config/rclone/rclone.conf', 'w').write(conf)

os.makedirs('/content/gdrive5tb', exist_ok=True)
# vfs-cache-mode full is required so torch.save / netCDF random writes work over the mount
subprocess.Popen(['rclone', 'mount', 'gdrive5tb:', '/content/gdrive5tb',
                  '--vfs-cache-mode', 'full', '--daemon'])
time.sleep(10)
assert os.path.ismount('/content/gdrive5tb') or os.listdir('/content/gdrive5tb') is not None, "mount failed"
print('mounted 5TB account ->', os.listdir('/content/gdrive5tb')[:5])

WORK = Path('/content/gdrive5tb/extremecast_prithvi')
(WORK/'merra-2').mkdir(parents=True, exist_ok=True)
(WORK/'ckpt').mkdir(parents=True, exist_ok=True)
print('persisting (5TB account) ->', WORK)

## 3 · Install Prithvi WxC

In [ ]:
import os, subprocess, sys
if not os.path.exists('/content/Prithvi-WxC'):
    subprocess.run(['git','clone','--depth','1','https://github.com/NASA-IMPACT/Prithvi-WxC.git',
                    '/content/Prithvi-WxC'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','/content/Prithvi-WxC','huggingface_hub'], check=True)
sys.path.insert(0,'/content/Prithvi-WxC')
print('installed')

## 4 · NASA Earthdata credentials
Entered at runtime; used only to set env vars for the downloader (not stored).

In [ ]:
import getpass, os
os.environ['GES_DISC_USER'] = input('Earthdata username: ')
os.environ['GES_DISC_PASSWORD'] = getpass.getpass('Earthdata password: ')
print('credentials set for this session')

## 5 · Configuration
`N_TRAIN_DATES` / `N_TEST_DATES` control how much MERRA-2 you download — raise them to push harder
(each date pulls several MERRA-2 files; raw goes to ephemeral disk, only compact inputs persist to Drive).
`LEAD_HOURS` is the forecast lead you fine-tune for.

In [ ]:
import numpy as np
surface_vars = ["EFLUX","GWETROOT","HFLUX","LAI","LWGAB","LWGEM","LWTUP","PS","QV2M","SLP",
                "SWGNT","SWTNT","T2M","TQI","TQL","TQV","TS","U10M","V10M","Z0M"]
static_surface_vars = ["FRACI","FRLAND","FROCEAN","PHIS"]
vertical_vars = ["CLOUD","H","OMEGA","PL","QI","QL","QV","T","U","V"]
levels = [34.,39.,41.,43.,44.,45.,48.,51.,53.,56.,63.,68.,71.,72.]
padding = {"level":(0,0), "lat":(0,-1), "lon":(0,0)}
positional_encoding = "fourier"
variable_names = surface_vars + [f"{v}_level_{l}" for v in vertical_vars for l in levels]
T2M_IDX = surface_vars.index("T2M")

INPUT_STEP_H = 6          # spacing between the 2 input frames
LEAD_HOURS   = 24         # forecast lead to fine-tune for

# HEATWAVE-relevant sampling: Northern-Hemisphere summer (JJA) across several years.
# With 5 TB of Drive, storage is NOT the limit -- push TRAIN_YEARS / lower STEP_DAYS to
# download & train on more. Download is restartable across sessions, so build it up over time.
TRAIN_YEARS = [2016, 2017, 2018, 2019]   # add years to scale up
TEST_YEARS  = [2020]                      # held-out year
MONTHS      = (6, 7, 8)                    # JJA (NH summer heat season)
STEP_DAYS   = 3                            # 1 = every day (max data), larger = fewer dates

def season_dates(years, months=MONTHS, step=STEP_DAYS):
    out = []
    for y in years:
        d, end = np.datetime64(f"{y}-{months[0]:02d}-01"), np.datetime64(f"{y}-{months[-1]:02d}-28")
        while d <= end:
            out.append(d); d = d + np.timedelta64(step, "D")
    return out

train_dates = season_dates(TRAIN_YEARS)
test_dates  = season_dates(TEST_YEARS)
print(f"train dates: {len(train_dates)}  test dates: {len(test_dates)}  lead {LEAD_HOURS}h "
      f"(JJA {TRAIN_YEARS} -> test {TEST_YEARS})")

## 6 · Download partial MERRA-2 (real) → Drive
Compact Prithvi inputs persist to your 5 TB Drive. **`download_dir` is intentionally omitted** so each
date's raw MERRA-2 goes to a temporary folder that is auto-deleted after formatting — otherwise raw files
pile up and fill Colab's ephemeral disk. Restartable: already-downloaded dates are skipped.

In [ ]:
from time import sleep
from tempfile import TemporaryDirectory
from PrithviWxC.download import get_prithvi_wxc_input

INP = WORK/'merra-2'

def ensure_dates(dates):
    for t in dates:
        stamp = str(t)[:10].replace('-','')
        if list(INP.glob(f'*{stamp}*')):        # already downloaded this day
            print('skip', stamp); continue
        for attempt in range(5):
            try:
                # Pass our OWN temp dir as download_dir: the package's internal TemporaryDirectory
                # path is broken (missing import), and this auto-cleans raw MERRA-2 each date.
                with TemporaryDirectory(dir='/content') as raw:
                    get_prithvi_wxc_input(t, input_time_step=INPUT_STEP_H, lead_time=LEAD_HOURS,
                                          input_data_dir=INP, download_dir=Path(raw))
                print('ok', stamp); break
            except Exception as e:
                print('retry', stamp, attempt, repr(e)[:200]); sleep(20*(attempt+1))
        else:
            print('FAILED', stamp)

ensure_dates(train_dates)
ensure_dates(test_dates)
print('formatted inputs on Drive:', len(list(INP.glob('*.nc'))))

## 7 · Model assets (weights + climatology + scalers) → Drive

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M"
DATA = str(WORK)
for f in ["musigma_surface.nc","musigma_vertical.nc","anomaly_variance_surface.nc","anomaly_variance_vertical.nc"]:
    hf_hub_download(repo_id=REPO, filename=f"climatology/{f}", local_dir=DATA)
hf_hub_download(repo_id=REPO, filename="config.yaml", local_dir=DATA)
hf_hub_download(repo_id=REPO, filename="prithvi.wxc.2300m.v1.pt", local_dir=f"{DATA}/weights")
print("assets ready on Drive")

## 8 · Datasets + scalers

In [ ]:
from PrithviWxC.dataloaders.merra2 import (Merra2Dataset, preproc,
    input_scalers, output_scalers, static_input_scalers)

clim = WORK/'climatology'
def make_ds(t0, t1):
    return Merra2Dataset(
        time_range=(str(t0), str(t1)),
        lead_times=[LEAD_HOURS], input_times=[-INPUT_STEP_H],
        data_path_surface=WORK/'merra-2', data_path_vertical=WORK/'merra-2',
        climatology_path_surface=WORK/'merra-2', climatology_path_vertical=WORK/'merra-2',
        surface_vars=surface_vars, static_surface_vars=static_surface_vars,
        vertical_vars=vertical_vars, levels=levels, positional_encoding=positional_encoding)

train_ds = make_ds(train_dates[0], train_dates[-1] + np.timedelta64(2,'D'))
test_ds  = make_ds(test_dates[0],  test_dates[-1]  + np.timedelta64(2,'D'))
assert len(train_ds) > 0 and len(test_ds) > 0, "no samples - check downloads / date ranges"
print("train samples", len(train_ds), "| test samples", len(test_ds))

sms=WORK/'climatology/musigma_surface.nc'; vms=WORK/'climatology/musigma_vertical.nc'
sos=WORK/'climatology/anomaly_variance_surface.nc'; vos=WORK/'climatology/anomaly_variance_vertical.nc'
in_mu, in_sig = input_scalers(surface_vars, vertical_vars, levels, sms, vms)
output_sig = output_scalers(surface_vars, vertical_vars, levels, sos, vos)
static_mu, static_sig = static_input_scalers(sms, static_surface_vars)

## 9 · Build Prithvi (frozen) + a trainable T2M correction head

In [ ]:
import yaml, torch.nn as nn
from PrithviWxC.model import PrithviWxC
device = torch.device('cuda')

with open(f"{WORK}/config.yaml") as f: p = yaml.safe_load(f)["params"]
model = PrithviWxC(
    in_channels=p["in_channels"], input_size_time=p["input_size_time"],
    in_channels_static=p["in_channels_static"],
    input_scalers_mu=in_mu, input_scalers_sigma=in_sig,
    input_scalers_epsilon=p["input_scalers_epsilon"],
    static_input_scalers_mu=static_mu, static_input_scalers_sigma=static_sig,
    static_input_scalers_epsilon=p["static_input_scalers_epsilon"],
    output_scalers=output_sig**0.5,
    n_lats_px=p["n_lats_px"], n_lons_px=p["n_lons_px"],
    patch_size_px=p["patch_size_px"], mask_unit_size_px=p["mask_unit_size_px"],
    mask_ratio_inputs=0.0, mask_ratio_targets=0.0,
    embed_dim=p["embed_dim"], n_blocks_encoder=p["n_blocks_encoder"],
    n_blocks_decoder=p["n_blocks_decoder"], mlp_multiplier=p["mlp_multiplier"],
    n_heads=p["n_heads"], dropout=p["dropout"], drop_path=p["drop_path"],
    parameter_dropout=p["parameter_dropout"], residual="climate",
    masking_mode="global", encoder_shifting=True, decoder_shifting=True,
    positional_encoding=positional_encoding, checkpoint_encoder=[], checkpoint_decoder=[])
state = torch.load(f"{WORK}/weights/prithvi.wxc.2300m.v1.pt", weights_only=False)
model.load_state_dict(state.get("model_state", state), strict=True)
model = model.to(device).eval()
for prm in model.parameters(): prm.requires_grad_(False)   # freeze backbone

class T2MHead(nn.Module):
    # Learns a residual correction to Prithvi's T2M forecast from its full output field.
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_ch,64,3,padding=1), nn.GELU(),
                                 nn.Conv2d(64,32,3,padding=1), nn.GELU(),
                                 nn.Conv2d(32,1,1))
    def forward(self, full_out): return full_out[:,T2M_IDX] + self.net(full_out).squeeze(1)

head = T2MHead(len(variable_names)).to(device)
print("trainable head params:", sum(x.numel() for x in head.parameters()))

## 10 · Extreme-aware loss + checkpoint/resume helpers
Loss up-weights hot anomalies so the head focuses on heat extremes.

In [ ]:
import torch
def heat_weighted_mse(pred, target, beta=1.5):
    a = target - target.mean()
    w = torch.exp(beta * (a / (a.std() + 1e-6)).clamp(min=0))   # emphasize hot tail
    w = w / w.mean().clamp_min(1e-6)
    return (w * (pred - target)**2).mean()

CKPT = WORK/'ckpt/head.pt'
def save_ckpt(step, epoch, opt):
    torch.save({'head':head.state_dict(),'opt':opt.state_dict(),'step':step,'epoch':epoch}, CKPT)
def load_ckpt(opt):
    if CKPT.exists():
        c = torch.load(CKPT, map_location=device)
        head.load_state_dict(c['head']); opt.load_state_dict(c['opt'])
        print(f"resumed from step {c['step']} epoch {c['epoch']}"); return c['step'], c['epoch']
    return 0, 0

## 11 · Train — runs until the time budget, checkpointing to Drive
Set `TIME_BUDGET_H` a bit under your session limit. Re-run this cell in a *new* session to **resume and
accumulate more epochs** (it reloads the checkpoint). This is how you "run it till Colab can take it."

In [ ]:
import time
from torch.utils.data import DataLoader
from PrithviWxC.dataloaders.merra2 import preproc

TIME_BUDGET_H = 11.0
CKPT_EVERY = 25
loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                    collate_fn=lambda b: preproc(b, padding))
opt = torch.optim.AdamW(head.parameters(), lr=3e-4, weight_decay=0.05)
step, epoch = load_ckpt(opt)
t0 = time.time(); head.train()
try:
    while time.time() - t0 < TIME_BUDGET_H*3600:
        for batch in loader:
            batch = {k:(v.to(device) if torch.is_tensor(v) else v) for k,v in batch.items()}
            with torch.no_grad():
                full = model(batch)                       # frozen backbone forecast
            pred = head(full)                             # corrected T2M
            loss = heat_weighted_mse(pred, batch['y'][:,T2M_IDX])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            step += 1
            if step % 5 == 0:
                print(f"epoch {epoch} step {step} loss {loss.item():.4f} "
                      f"elapsed {(time.time()-t0)/60:.1f}m", flush=True)
            if step % CKPT_EVERY == 0:
                save_ckpt(step, epoch, opt); print("  checkpointed ->", CKPT)
            if time.time() - t0 >= TIME_BUDGET_H*3600: break
        epoch += 1
finally:
    save_ckpt(step, epoch, opt)
    print(f"stopped at step {step}, epoch {epoch}; checkpoint saved to Drive.")

## 12 · Evaluate: persistence vs Prithvi zero-shot vs Prithvi + fine-tuned head

In [ ]:
import numpy as np
def area_weights(nlat):
    lat = np.linspace(-90, 90, nlat); w = np.cos(np.deg2rad(lat)); return w/w.mean()

def wrmse(p, o, wlat):
    d = (p - o)**2
    return float(np.sqrt((d * wlat[None,:,None]).mean()))

head.eval()
te = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=lambda b: preproc(b, padding))
per_e=[]; zs_e=[]; ft_e=[]
with torch.no_grad():
    for batch in te:
        batch = {k:(v.to(device) if torch.is_tensor(v) else v) for k,v in batch.items()}
        y = batch['y'][:,T2M_IDX]
        persist = batch['x'][:,-1,T2M_IDX]        # last input frame T2M
        full = model(batch); zs = full[:,T2M_IDX]; ft = head(full)
        w = area_weights(y.shape[-2])
        per_e.append(wrmse(persist.cpu().numpy(), y.cpu().numpy(), w))
        zs_e.append(wrmse(zs.cpu().numpy(),      y.cpu().numpy(), w))
        ft_e.append(wrmse(ft.cpu().numpy(),      y.cpu().numpy(), w))
print(f"T2M RMSE @ {LEAD_HOURS}h  (lower is better)")
print(f"  persistence      : {np.mean(per_e):.3f}")
print(f"  Prithvi zero-shot: {np.mean(zs_e):.3f}")
print(f"  Prithvi + head   : {np.mean(ft_e):.3f}")

## 13 · Presenting this honestly
- Report the **data window** (how many init dates / what period you actually downloaded) and that it's a
  **frozen-backbone downstream fine-tune**, not full pretraining.
- The headline is the **three-way comparison** (persistence / Prithvi zero-shot / Prithvi+head) on held-out dates,
  ideally with the **heat-extreme subset** highlighted.
- Cross-reference the ExtremeCast **ERA5 baseline** (persistence/damped/small transformer) for context.
- To push further: more init dates (Cell 5), more epochs (re-run Cell 11 across sessions), or LoRA on the
  backbone instead of a head (bigger lift, needs A100 + gradient checkpointing).

**Limits that remain:** Colab sessions time out and disk is ephemeral (hence Drive); full-record, full-model
training is an HPC/persistent-cloud job. This gives a real, defensible result within Colab's envelope.